# 第 15 章　文本分析
## jieba、LDA、情感分析与 LLM API

::: {.callout-note}
## 本章要点

1. **中文分词**：`jieba` 分词、停用词、TF-IDF
2. **主题模型**：LDA（`gensim`），主题数选择
3. **情感分析**：金融词典（LM / NTUSD）vs FinBERT-CN
4. **LLM API**：DeepSeek / OpenAI 批量提取年报结构化信息，成本控制
5. **综合案例**：管理层语调指数 → 与财务数据合并 → 面板回归
:::


In [ ]:
# ── 第 15 章　文本分析　──────────────────────────────────────────────
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import jieba

warnings.filterwarnings('ignore')
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
OUTPUT = 'output'; os.makedirs(OUTPUT, exist_ok=True)
print('环境就绪 ✓')


---

## 1　中文分词与 TF-IDF

与英文不同，中文文本没有天然的空格分隔词语，
必须先做**分词（Tokenization）**再做后续分析。
`jieba` 是最常用的中文分词库。


In [ ]:
# ── 1.1  jieba 分词基础 ─────────────────────────────────────────────
# 模拟年报摘要文本
texts = [
    '本公司坚持创新驱动战略，持续加大研发投入，积极推进数字化转型，提升核心竞争力。',
    '受宏观经济下行压力影响，公司营业收入出现下滑，净利润同比减少，经营压力加大。',
    '公司持续完善公司治理结构，强化内部控制，积极回报股东，全年现金分红稳定增长。',
    '公司把握新能源行业发展机遇，加快产能扩张，市场份额稳步提升，盈利能力显著改善。',
    '受原材料价格大幅上涨影响，公司毛利率有所下降，成本压力持续上升。',
]

# 停用词（简化版）
stopwords = {'的','是','在','和','了','公司','本','受','有','对'}

def tokenize(text, stop=stopwords):
    tokens = jieba.cut(text, cut_all=False)
    return [t for t in tokens if t.strip() and t not in stop and len(t) > 1]

tokenized = [tokenize(t) for t in texts]
for i, toks in enumerate(tokenized):
    print(f'文本 {i+1}：{" / ".join(toks)}')


In [ ]:
# ── 1.2  TF-IDF 关键词提取 ──────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer

# 将分词结果重新连接（用空格分隔）
corpus = [' '.join(toks) for toks in tokenized]

vectorizer = TfidfVectorizer(max_features=20, min_df=1)
tfidf_mat  = vectorizer.fit_transform(corpus)
vocab      = vectorizer.get_feature_names_out()

# 每篇文档的 Top-3 关键词
for i, text_id in enumerate(range(len(texts))):
    row    = tfidf_mat[text_id].toarray()[0]
    top3   = [(vocab[j], row[j]) for j in row.argsort()[::-1][:3] if row[j] > 0]
    kws    = ', '.join([f'{w}({s:.3f})' for w, s in top3])
    print(f'文本 {i+1} 关键词：{kws}')


---

## 2　LDA 主题模型

**LDA（Latent Dirichlet Allocation）**假设每篇文档
是多个主题的混合，每个主题是词语的概率分布。
通过训练，可以发现语料库中潜在的主题结构。


In [ ]:
# ── 2.1  LDA 主题模型（gensim）─────────────────────────────────────
try:
    from gensim import corpora
    from gensim.models import LdaModel
    GENSIM_OK = True
except ImportError:
    GENSIM_OK = False
    print('gensim 未安装（pip install gensim）')

if GENSIM_OK:
    # 构建词典和语料
    dictionary = corpora.Dictionary(tokenized)
    dictionary.filter_extremes(no_below=1, no_above=0.9)
    corpus_bow = [dictionary.doc2bow(toks) for toks in tokenized]

    # 训练 LDA（主题数 = 2，因样本小）
    lda = LdaModel(
        corpus_bow, num_topics=2, id2word=dictionary,
        passes=20, random_state=42, alpha='auto')

    print('LDA 主题：')
    for tid, words in lda.print_topics(num_words=6):
        print(f'  主题 {tid+1}：{words}')

    # 每篇文档的主题分布
    print('\n文档-主题分布：')
    for i, doc in enumerate(corpus_bow):
        dist = dict(lda.get_document_topics(doc))
        main = max(dist, key=dist.get)
        print(f'  文本 {i+1}：主题 {main+1}（概率 {dist[main]:.3f}）')
else:
    print('跳过 LDA（gensim 未安装）')


---

## 3　情感分析

### 3.1　词典方法 vs 预训练模型

| 方法 | 工具 | 优点 | 缺点 |
|------|------|------|------|
| **词典方法** | LM（Loughran-McDonald）/ NTUSD | 可解释，计算快 | 无法理解语境 |
| **预训练模型** | FinBERT-CN / ERNIE-Finance | 理解语境，精度高 | 计算慢，需 GPU |
| **LLM API** | GPT-4 / DeepSeek | 最灵活，无需微调 | 成本高，有速率限制 |


In [ ]:
# ── 3.1  词典情感分析 ────────────────────────────────────────────────
# 构建简化版中文金融情感词典
positive_words = {'增长','提升','改善','优化','强劲','盈利',
                   '超预期','创新','机遇','稳定','回暖','突破'}
negative_words = {'下滑','亏损','压力','风险','下降','减少',
                   '困难','挑战','下行','低迷','违约','损失'}

def sentiment_score(text):
    '''词典情感得分：正词数 - 负词数（归一化）'''
    tokens = set(tokenize(text))
    pos = len(tokens & positive_words)
    neg = len(tokens & negative_words)
    total = max(pos + neg, 1)
    return (pos - neg) / total

print('词典情感分析结果：')
for i, text in enumerate(texts):
    score = sentiment_score(text)
    label = '正面' if score > 0 else '负面' if score < 0 else '中性'
    print(f'  文本 {i+1}：{score:+.3f}（{label}）')


In [ ]:
# ── 4.1  LLM API 批量提取年报信息 ────────────────────────────────────
# 演示如何用 LLM API（以 DeepSeek 为例）批量从年报文本提取结构化信息
# 真实使用时替换 API_KEY，并注意成本控制（见 4.3 节）

PROMPT_TEMPLATE = '''
请从以下年报段落中提取结构化信息，以 JSON 格式输出，包含以下字段：
- tone: 整体语调，取值为 positive/neutral/negative
- forward_looking: 是否包含前瞻性陈述（true/false）
- key_risks: 提到的主要风险（列表，最多 3 条，每条不超过 15 字）
- growth_targets: 提到的增长目标（列表，可为空）

年报段落：
{text}

只输出 JSON，不要有任何额外说明。
'''

def call_llm_api(text, api_key='your_api_key', model='deepseek-chat'):
    '''
    调用 LLM API 提取结构化信息。
    实际使用时替换 API_KEY 和 base_url。
    '''
    import requests
    prompt = PROMPT_TEMPLATE.format(text=text)
    payload = {
        'model': model,
        'messages': [{'role': 'user', 'content': prompt}],
        'temperature': 0.1,   # 低温度保证输出稳定
        'max_tokens': 200,
    }
    # 实际调用（注释掉以避免意外消费）
    # resp = requests.post(
    #     'https://api.deepseek.com/v1/chat/completions',
    #     json=payload,
    #     headers={'Authorization': f'Bearer {api_key}'},
    #     timeout=30
    # )
    # return json.loads(resp.json()['choices'][0]['message']['content'])
    return None

# 模拟 LLM 输出（演示 JSON 解析逻辑）
mock_outputs = [
    '{"tone": "positive", "forward_looking": true, "key_risks": [], "growth_targets": ["持续加大研发投入"]}',
    '{"tone": "negative", "forward_looking": false, "key_risks": ["宏观经济下行", "营收下滑"], "growth_targets": []}',
    '{"tone": "neutral",  "forward_looking": false, "key_risks": [], "growth_targets": ["分红稳定增长"]}',
]

print('LLM API 结构化提取结果（模拟）：')
for i, output in enumerate(mock_outputs):
    try:
        parsed = json.loads(output)
        print(f'文本 {i+1}：语调={parsed["tone"]}  前瞻性={parsed["forward_looking"]}',
              f' 风险={parsed["key_risks"]}')
    except json.JSONDecodeError as e:
        print(f'文本 {i+1}：解析失败（{e}）')


### 4.2　成本控制策略

| 策略 | 说明 |
|------|------|
| **批处理（Batch API）**| OpenAI/DeepSeek 批量 API 通常比单次调用便宜 50% |
| **缓存结果**| 把已提取的结果存入数据库，避免重复计费 |
| **分层处理**| 先用词典法快速过滤，只对「不确定」的样本用 LLM |
| **控制 token 数**| `max_tokens` 设置合理上限；Prompt 避免冗余文字 |
| **小样本测试**| 正式批量前先用 20 条验证输出格式是否稳定 |


---

## 5　综合案例：管理层语调指数 → 面板回归

把本章的文本分析结果和前几章的财务面板数据整合，
形成一个完整的「文本 → 量化 → 回归」研究流程。


In [ ]:
# ── 5.1  综合案例：管理层语调指数与盈利能力的关系 ──────────────────
import numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression

# 模拟面板数据（包含语调得分）
N_FIRMS = 100; YEARS = list(range(2018, 2024))
RNG2 = np.random.default_rng(99)

records = []
for firm_id in range(N_FIRMS):
    firm_quality = RNG2.normal(0, 1)  # 公司固定特征
    for yr in YEARS:
        tone  = 0.3 * firm_quality + RNG2.normal(0, 0.5)  # 语调受公司质量影响
        roa   = 0.05 + 0.4 * firm_quality + 0.2 * tone + RNG2.normal(0, 0.02)
        records.append({'firm': f'F{firm_id:03d}', 'year': yr,
                         'tone': tone, 'roa': roa,
                         'size': RNG2.normal(23, 1.5)})

df_panel = pd.DataFrame(records)

# 按第 5-6 章的方法做面板回归
try:
    import pyfixest as pf
    m1 = pf.feols('roa ~ tone + size', data=df_panel, vcov={'CRV1':'firm'})
    m2 = pf.feols('roa ~ tone + size | firm + year', data=df_panel,
                   vcov={'CRV1':'firm'})
    pf.etable([m1, m2], coef_fmt='b\n(se)', digits=3, stars=True,
              labels={'roa':'ROA','tone':'管理层语调','size':'公司规模'})
    print('\n注：加入公司 FE 后语调系数变化，说明公司固定特征是混淆变量')
except ImportError:
    print('（pyfixest 未安装，跳过面板回归演示）')
    m_ols = pd.DataFrame({'变量':['tone','size'],'coef':[0.198, 0.031],'p':[0.000,0.021]})
    print('OLS 结果（示意）：')
    print(m_ols.to_string(index=False))


---

## 6　章末练习

**练习 1（TF-IDF 关键词）**
对 CSMAR（或 AKShare）获取的 10 份上市公司年报摘要，
用 jieba + TF-IDF 提取每份年报的 Top-10 关键词，
用词云图（`pip install wordcloud`）可视化整体语料库的高频词。

**练习 2（LDA 主题数选择）**
对年报语料，用困惑度（Perplexity）和主题一致性（Coherence Score）
分别评估 $k = 3, 5, 8, 10$ 个主题的 LDA 模型，
选出最优主题数，解释每个主题的含义。

**练习 3（情感 → 回归）**
计算每家公司每年的年报语调得分（词典法），
合并至财务面板数据（第 4 章），
用第 6 章的 pyfixest 做面板回归，
检验「语调更正面的公司次年 ROA 是否更高」，
讨论因果解释的局限性。

**练习 4（LLM API 设计）**
设计一个 Prompt，用于从上市公司借贷公告中提取：
贷款银行、金额、利率、期限、是否为关联方贷款。
写出 Prompt 文本、期望的 JSON 输出格式、
以及验证输出正确性的测试用例（3 条）。
